---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# 12. Directional Data — Extraction and Consolidation

## Objective

Read the directional survey files (`*_direc.txt`) from each well and consolidate
into a single file `data/raw/directionals/directional_wells.csv` for use in
Notebook 14.

> ⚠️ **Reproducibility note:** This notebook requires access to raw directionala
> survey files stored on the LABAP internal server, which are not publicly available.
> The consolidated output (`data/raw/directionals/directional_wells.csv`) is already
> provided in the repository — **there is no need to run this notebook to reproduce
> the analyses in notebooks 13 and 14.**a

## Generated Columns

| Column | Description |
|---|---|
| `Well_Name` | Well name (uppercase, same as main dataset) |
| `MD` | Measured depth (m) — join key with the main dataset |
| `TVD` | True vertical depth (m) |
| `INCL` | Inclination (degrees) |
| `AZIM` | Azimuth (degrees) |
| `NS` | North/South offset (m), + = North |
| `EW` | East/West offset (m), + = East |
| `MD_TVD_diff` | MD − TVD difference (m) |
| `Is_Deviated` | True if inclination ≥ 5° at that point |

## 1. Imports

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.graph_objects as go

## 2. Configuration

In [ ]:
# Paths
POCOS_ROOT    = Path('Z:/SERGIPE/POCOS')
SURVEY_FOLDER = 'Dados Direcionais'
OUTPUT_CSV    = Path('../data/raw/directionals/directional_wells.csv')
FIGURES_DIR   = Path('../results/directional')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Wells outside the study scope
EXCLUDE = {
    '1-BRSA-1141-SES', '_3-BRSA-1368-SES',
    'LAS', 'LAS_corrigido_2026', 'LAS_output_IP', 'checkshots'
}

# Inclination threshold to classify a point as deviated
DEVI_THRESHOLD = 5.0

print(f'Well root      : {POCOS_ROOT}')
print(f'CSV output     : {OUTPUT_CSV}')
print(f'Figures dir    : {FIGURES_DIR}')

## 3. Survey File Parser

Reads a `*_direc.txt` file and returns a DataFrame with columns
`MD, INCL, TVD, AZIM, NS, EW`.
Ignores the bearing field in parentheses (e.g. `(N44.52E)`) and skips
header lines until it finds the line containing `PROFUNDIDADE` or `MEDIDA`.

In [ ]:
def parse_survey_file(filepath: Path) -> pd.DataFrame | None:
    """
    Reads a directional survey file and returns a DataFrame with columns
    MD, INCL, TVD, AZIM, NS, EW.
    Returns None if no valid data is found.
    """
    rows = []
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    data_started = False
    for line in lines:
        line = line.strip()
        if not line or line.startswith('-') or line.startswith('_'):
            continue
        if 'PROFUNDIDADE' in line.upper() or 'MEDIDA' in line.upper():
            data_started = True
            continue
        if not data_started:
            continue

        # Remove bearing field in parentheses: e.g. (N44.52E)
        line_clean = re.sub(r'\([^)]*\)', '', line)
        parts = line_clean.split()
        if len(parts) < 6:
            continue

        try:
            rows.append({
                'MD'  : float(parts[0]),
                'INCL': float(parts[1]),
                'TVD' : float(parts[2]),
                'AZIM': float(parts[3]),
                'NS'  : float(parts[4]),
                'EW'  : float(parts[5]),
            })
        except (ValueError, IndexError):
            continue

    if not rows:
        return None

    df = pd.DataFrame(rows)

    # If NS/EW are absolute UTM coordinates (values > 10,000 m)
    # convert to relative offset by subtracting the first station
    if abs(df['NS'].iloc[0]) > 10000 or abs(df['EW'].iloc[0]) > 10000:
        df['NS'] = df['NS'] - df['NS'].iloc[0]
        df['EW'] = df['EW'] - df['EW'].iloc[0]

    return df

## 4. Survey Collection

In [ ]:
all_surveys = []
n_ok  = 0
n_err = 0

print(f'Scanning: {POCOS_ROOT}')
print('=' * 65)

for well_dir in sorted(POCOS_ROOT.iterdir()):
    if not well_dir.is_dir() or well_dir.name in EXCLUDE:
        continue

    survey_dir = well_dir / SURVEY_FOLDER
    if not survey_dir.exists():
        continue

    survey_files = (
        list(survey_dir.glob('*direc*.txt'))     +
        list(survey_dir.glob('*survey*.txt'))    +
        list(survey_dir.glob('*direcional*.txt')) +
        list(survey_dir.glob('*.txt'))
    )
    if not survey_files:
        print(f'  ⚠️  {well_dir.name}: folder exists but no .txt files found')
        n_err += 1
        continue

    survey_file = max(survey_files, key=lambda f: f.stat().st_size)
    df_tmp = parse_survey_file(survey_file)

    if df_tmp is None or df_tmp.empty:
        print(f'  ⚠️  {well_dir.name}: no parseable data ({survey_file.name})')
        n_err += 1
        continue

    well_name = well_dir.name.upper()
    df_tmp.insert(0, 'Well_Name', well_name)
    all_surveys.append(df_tmp)
    n_ok += 1

    print(f'  ✅ {well_name:<30s} {len(df_tmp):3d} stations | '
          f'MD={df_tmp["MD"].max():6.0f}m | max_incl={df_tmp["INCL"].max():.1f}°')

print('=' * 65)
print(f'Successfully loaded    : {n_ok}')
print(f'With errors            : {n_err}')

## 5. Consolidation and Derived Columns

In [ ]:
if not all_surveys:
    raise RuntimeError('No survey loaded. Check the paths.')

df_dir = pd.concat(all_surveys, ignore_index=True)

# Derived columns
df_dir['MD_TVD_diff'] = (df_dir['MD'] - df_dir['TVD']).round(3)
df_dir['Is_Deviated'] = df_dir['INCL'] >= DEVI_THRESHOLD
df_dir = df_dir.sort_values(['Well_Name', 'MD']).reset_index(drop=True)

print(f'Total rows      : {len(df_dir):,}')
print(f'Wells included  : {df_dir["Well_Name"].nunique()}')
print(f'Columns         : {list(df_dir.columns)}')

## 6. Per-Well Summary

In [ ]:
df_dir_summary = df_dir.groupby('Well_Name').agg(
    N_stations      = ('MD',          'count'),
    MD_min          = ('MD',          'min'),
    MD_max          = ('MD',          'max'),
    TVD_max         = ('TVD',         'max'),
    Max_INCL        = ('INCL',        'max'),
    Max_MD_TVD_diff = ('MD_TVD_diff', 'max'),
    NS_final        = ('NS',          'last'),
    EW_final        = ('EW',          'last'),
    Pct_Deviated    = ('Is_Deviated', 'mean'),
).reset_index()

df_dir_summary['Pct_Deviated'] = (df_dir_summary['Pct_Deviated'] * 100).round(1)
df_dir_summary['Desl_H']       = np.sqrt(
    df_dir_summary['NS_final']**2 + df_dir_summary['EW_final']**2
).round(1)
df_dir_summary['Classification'] = df_dir_summary['Max_INCL'].apply(
    lambda v: 'Directional (>20°)' if v > 20
    else ('Slight deviation (5–20°)' if v > 5 else 'Vertical (≤5°)')
)

print('PER-WELL SUMMARY')
print('=' * 95)
print(df_dir_summary[['Well_Name','Max_INCL','MD_max','TVD_max',
                       'NS_final','EW_final','Desl_H','Classification']].to_string(index=False))

## 7. Directional Well Trajectory Visualization

Three independent visualizations — only wells with maximum inclination ≥ 5°:
- **Side View** (Matplotlib): Horizontal Displacement × TVD
- **Top View / Plan** (Matplotlib): East × North
- **Interactive 3D View** (Plotly): EW × NS × TVD with free rotation

In [ ]:
# Filter only deviated wells
dir_wells = df_dir_summary[df_dir_summary['Max_INCL'] >= 5.0]['Well_Name'].tolist()
df_plot   = df_dir[df_dir['Well_Name'].isin(dir_wells)].copy()
df_plot['Desl_H'] = np.sqrt(df_plot['NS']**2 + df_plot['EW']**2)

colors    = plt.cm.tab10(np.linspace(0, 1, len(dir_wells)))
color_map = {w: colors[i] for i, w in enumerate(sorted(dir_wells))}

print(f'Deviated wells (incl ≥ 5°): {len(dir_wells)}')
for w in sorted(dir_wells):
    incl = df_dir_summary.loc[df_dir_summary['Well_Name'] == w, 'Max_INCL'].values[0]
    print(f'  {w:<35s} max_incl={incl:.1f}°')

In [ ]:
# Side View: Desl_H × TVD
fig, ax = plt.subplots(figsize=(10, 8))

for well in sorted(dir_wells):
    wdf   = df_plot[df_plot['Well_Name'] == well].sort_values('MD')
    incl  = df_dir_summary.loc[df_dir_summary['Well_Name'] == well, 'Max_INCL'].values[0]
    label = f'{well.replace("-SES", "")} (max {incl:.1f}°)'
    color = color_map[well]

    ax.plot(wdf['Desl_H'], -wdf['TVD'], color=color, linewidth=2, label=label)
    ax.annotate('', xy=(wdf['Desl_H'].iloc[-1], -wdf['TVD'].iloc[-1]),
                xytext=(wdf['Desl_H'].iloc[-2], -wdf['TVD'].iloc[-2]),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))

ax.set_xlabel('Horizontal Displacement (m)', fontsize=12)
ax.set_ylabel('TVD (m)', fontsize=12)
ax.set_title('Side View — Directional Well Trajectories\nSergipe-Alagoas Basin',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
# ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'trajetoria_vista_lateral.png', dpi=150, bbox_inches='tight')
plt.show()
print('Side view saved.')

In [ ]:
# Top View (Plan): EW × NS
fig, ax = plt.subplots(figsize=(9, 9))

for well in sorted(dir_wells):
    wdf   = df_plot[df_plot['Well_Name'] == well].sort_values('MD')
    incl  = df_dir_summary.loc[df_dir_summary['Well_Name'] == well, 'Max_INCL'].values[0]
    label = f'{well.replace("-SES", "")} (max {incl:.1f}°)'
    color = color_map[well]

    ax.plot(wdf['EW'], wdf['NS'], color=color, linewidth=2, label=label)
    ax.scatter(wdf['EW'].iloc[0],  wdf['NS'].iloc[0],  color=color, s=60,  zorder=5, marker='o')
    ax.scatter(wdf['EW'].iloc[-1], wdf['NS'].iloc[-1], color=color, s=100, zorder=5, marker='v')
    ax.annotate(well.replace('-SES', ''),
                xy=(wdf['EW'].iloc[-1], wdf['NS'].iloc[-1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, color=color, fontweight='bold')

ax.scatter(0, 0, color='black', s=80, zorder=6, marker='+', linewidths=2)
ax.set_xlabel('East (m)', fontsize=12)
ax.set_ylabel('North (m)', fontsize=12)
ax.set_title('Top View (Plan) — Directional Well Trajectories\nSergipe-Alagoas Basin',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'trajetoria_vista_planta.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top view saved.')

In [ ]:
# Interactive 3D View
plotly_colors = [
    '#1f77b4', '#2ca02c', '#9467bd', '#d62728',
    '#8c564b', '#e377c2', '#17becf', '#bcbd22'
]

fig3d = go.Figure()

for i, well in enumerate(sorted(dir_wells)):
    wdf  = df_plot[df_plot['Well_Name'] == well].sort_values('MD')
    incl = df_dir_summary.loc[df_dir_summary['Well_Name'] == well, 'Max_INCL'].values[0]
    col  = plotly_colors[i % len(plotly_colors)]
    name = f'{well.replace("-SES", "")} ({incl:.1f}°)'

    fig3d.add_trace(go.Scatter3d(
        x=wdf['EW'].tolist(),
        y=wdf['NS'].tolist(),
        z=(-wdf['TVD']).tolist(),
        mode='lines+markers',
        name=name,
        line=dict(color=col, width=4),
        marker=dict(size=3, color=col),
        hovertemplate=(
            f'<b>{name}</b><br>EW: %{{x:.0f}} m<br>'
            'NS: %{y:.0f} m<br>TVD: %{customdata:.0f} m<br>'
            'MD: %{text} m<extra></extra>'
        ),
        customdata=wdf['TVD'].tolist(),
        text=wdf['MD'].round(0).astype(int).astype(str).tolist(),
    ))

    # Final point
    fig3d.add_trace(go.Scatter3d(
        x=[wdf['EW'].iloc[-1]], y=[wdf['NS'].iloc[-1]],
        z=[(-wdf['TVD']).iloc[-1]],
        mode='markers+text',
        marker=dict(size=8, color=col, symbol='diamond'),
        text=[well.replace('-SES', '')],
        textposition='top center',
        textfont=dict(size=10),
        showlegend=False, hoverinfo='skip',
    ))

fig3d.update_layout(
    title=dict(
        text='3D View — Directional Well Trajectories<br>'
             '<sup>Sergipe-Alagoas Basin | Official survey</sup>',
        font=dict(size=14), x=0.5
    ),
    scene=dict(
        xaxis_title='East (m)',
        yaxis_title='North (m)',
        zaxis_title='TVD (m)',
        xaxis=dict(showgrid=True, gridcolor='lightgray'),
        yaxis=dict(showgrid=True, gridcolor='lightgray', autorange='reversed'),
        zaxis=dict(showgrid=True, gridcolor='lightgray', autorange=True),
        aspectmode='manual',
        aspectratio=dict(x=1.5, y=1.5, z=1.0),
        camera=dict(eye=dict(x=1.5, y=-1.5, z=0.8)),
    ),
    legend=dict(title='Well (max incl.)', x=0.02, y=0.98,
                bgcolor='rgba(255,255,255,0.8)', bordercolor='gray',
                borderwidth=1, font=dict(size=10)),
    width=1200, height=800,
    margin=dict(l=0, r=0, t=60, b=0),
)

html_path = FIGURES_DIR / 'trajetoria_3d_interativa.html'
fig3d.write_html(str(html_path))
print(f'3D view saved to: {html_path}')
fig3d.show()

## 8. Save CSV

In [ ]:
df_dir.to_csv(OUTPUT_CSV, index=False)
print(f'✅ Salvo em: {OUTPUT_CSV}')
print(f'   Rows    : {len(df_dir):,}')
print(f'   Wells   : {df_dir["Well_Name"].nunique()}')